# theLook Analytics · 02 · dbt e MetricFlow

Questo notebook ricostruisce staging e mart con **dbt** a partire dalla cartella `dbt/` del repository, esegue tutti i test (compreso il controllo anomalie sui ricavi settimanali) e interroga le metriche del **semantic layer** con **MetricFlow**. Alla fine confronta i numeri con il report Power BI: devono coincidere, perché le definizioni sono le stesse.

- dbt scrive nei dataset `dbt_staging` e `dbt_marts`, separati dalle viste create a mano negli script SQL.
- Nessuna installazione sul computer: tutto gira in Google Colab, con lo stesso account Google del progetto BigQuery.
- Serve che il repository sia pubblico su GitHub.

**Come eseguirlo**: scrivi l'ID del progetto e l'indirizzo del repository nella prima cella, poi *Runtime → Esegui tutto*.

In [1]:
PROJECT_ID = "thelook-analytics-514008"
GITHUB_REPO = "https://github.com/kejSirio/thelook-analytics.git"

if PROJECT_ID.startswith("scrivi-qui"):
    raise ValueError("Scrivi l'ID del progetto Google Cloud in PROJECT_ID, poi riesegui il notebook.")
if "TUO-UTENTE" in GITHUB_REPO:
    raise ValueError("Scrivi l'indirizzo del tuo repository GitHub in GITHUB_REPO, poi riesegui il notebook.")

In [2]:
# Accesso a BigQuery con l'account Google: dbt e MetricFlow useranno queste credenziali
from google.colab import auth
auth.authenticate_user()

## 1 · Installazione

dbt e MetricFlow vanno in un ambiente Python 3.12 separato, creato con `uv`, così non dipendono dalla versione di Python di Colab. Ci vuole circa un minuto.

In [3]:
import os

!pip install -q uv
!uv venv -q --python 3.12 /content/dbt-env
!uv pip install -q --python /content/dbt-env/bin/python "dbt-metricflow[dbt-bigquery]"

os.environ["PATH"] = "/content/dbt-env/bin:" + os.environ["PATH"]
!dbt --version

Core:
  - installed: 1.12.6
  - latest:    1.12.6 - Up to date!

Plugins:
  - bigquery: 1.12.2 - Up to date!


10:46:19  dbt v2 is available, and is the default for new installations. This is dbt Core v1.
          New project - install v2: https://docs.getdbt.com/docs/local/install-dbt?utm_source=dbt-cli
          Existing v1 project - upgrade guide: https://docs.getdbt.com/docs/dbt-versions/dbt-upgrade/upgrading-to-v2?utm_source=dbt-cli


In [4]:
import shutil

# Copia aggiornata del repository
if os.path.exists("/content/repo"):
    shutil.rmtree("/content/repo")
!git clone -q {GITHUB_REPO} /content/repo
%cd /content/repo/dbt

# Profilo di connessione: OAuth con le credenziali di Colab, location US come i dati pubblici
profilo = f"""thelook:
  target: dev
  outputs:
    dev:
      type: bigquery
      method: oauth
      project: {PROJECT_ID}
      dataset: dbt
      location: US
      threads: 4
"""
# Il profilo va nella cartella del progetto: dbt lo trova lì, e MetricFlow
# lo cerca solo lì (o in DBT_PROFILES_DIR)
with open("profiles.yml", "w") as f:
    f.write(profilo)
os.environ["DBT_PROFILES_DIR"] = os.getcwd()

!dbt debug

/content/repo/dbt
10:46:28  Running with dbt=1.12.6
10:46:28  dbt version: 1.12.6
10:46:28  python version: 3.12.3
10:46:28  python path: /content/dbt-env/bin/python
10:46:28  os info: Linux-6.6.159+-x86_64-with-glibc2.39
10:46:30  Using profiles dir at /content/repo/dbt
10:46:30  Using profiles.yml file at /content/repo/dbt/profiles.yml
10:46:30  Using dbt_project.yml file at /content/repo/dbt/dbt_project.yml
10:46:30  adapter type: bigquery
10:46:30  adapter version: 1.12.2
10:46:30  Configuration:
10:46:30    profiles.yml file [OK found and valid]
10:46:30    dbt_project.yml file [OK found and valid]
10:46:30  Required dependencies:
10:46:30   - git [OK found]

10:46:30  Connection:
10:46:30    method: oauth
10:46:30    database: thelook-analytics-514008
10:46:30    execution_project: thelook-analytics-514008
10:46:30    schema: dbt
10:46:30    location: US
10:46:30    priority: None
10:46:30    maximum_bytes_billed: None
10:46:30    reservation: None
10:46:30    impersonate_service

## 2 · Build e test

`dbt build` carica il seed dei canali, crea i modelli nell'ordine giusto e dopo ciascuno esegue i suoi test. Nel riepilogo finale:

- **PASS**: test superato.
- **WARN**: avviso che non blocca, per esempio il controllo anomalie sulle settimane o uno stato d'ordine inatteso.
- **ERROR** o **FAIL**: va guardato prima di proseguire.

In [5]:
!dbt build

10:46:37  Running with dbt=1.12.6
10:46:40  Registered adapter: bigquery=1.12.2
10:46:40  Unable to do partial parsing because saved manifest not found. Starting full parse.
10:46:45  Found 12 models, 41 data tests, 1 seed, 5 sources, 14 metrics, 561 macros, 4 semantic models
10:46:45  
10:46:45  Concurrency: 4 threads (target='dev')
10:46:45  
10:46:46  1 of 54 START sql table model dbt_marts.dim_date ............................... [RUN]
10:46:46  2 of 54 START sql table model dbt_marts.metricflow_time_spine .................. [RUN]
10:46:46  3 of 54 START sql view model dbt_staging.stg_events ............................ [RUN]
10:46:46  4 of 54 START sql view model dbt_staging.stg_order_items ....................... [RUN]
10:46:47  3 of 54 OK created sql view model dbt_staging.stg_events ....................... [CREATE VIEW (0 processed) in 1.16s]
10:46:47  5 of 54 START sql view model dbt_staging.stg_orders ............................ [RUN]
10:46:47  4 of 54 OK created sql view mo

### Dettaglio di errori e avvisi

Per ogni test non superato, questa cella rilancia la query del test e mostra le righe che lo hanno fatto scattare. Per il controllo anomalie sono le settimane segnalate, con lo scarto rispetto alla media delle quattro precedenti.

In [6]:
import json
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)
risultati = json.load(open("target/run_results.json"))["results"]
segnalati = [r for r in risultati if r["status"] in ("fail", "warn", "error")]

if not segnalati:
    print("Nessun errore e nessun avviso.")
for r in segnalati:
    print(f"\n{r['status'].upper()} · {r['unique_id'].split('.')[2]} · {r.get('failures')} righe")
    sql = r.get("compiled_code")
    if sql:
        display(bq.query(sql).to_dataframe(create_bqstorage_client=False).head(20))

Nessun errore e nessun avviso.


## 3 · Semantic layer

`mf validate-configs` controlla che semantic model e metriche siano coerenti tra loro e con i dati in BigQuery.

In [7]:
!mf validate-configs

(To see warnings and future-errors, run again with flag `--show-all`)
✔ 🎉 Successfully parsed manifest from dbt project
✔ 🎉 Successfully validated the semantics of built manifest (ERRORS: 0, FUTURE_ERRORS: 0, WARNINGS: 0)
✔ 🎉 Successfully validated semantic models against data warehouse (ERRORS: 0, FUTURE_ERRORS: 0, WARNINGS: 0)
✔ 🎉 Successfully validated dimensions against data warehouse (ERRORS: 0, FUTURE_ERRORS: 0, WARNINGS: 0)
✔ 🎉 Successfully validated entities against data warehouse (ERRORS: 0, FUTURE_ERRORS: 0, WARNINGS: 0)
✔ 🎉 Successfully validated measures against data warehouse (ERRORS: 0, FUTURE_ERRORS: 0, WARNINGS: 0)
✔ 🎉 Successfully validated metrics against data warehouse (ERRORS: 0, FUTURE_ERRORS: 0, WARNINGS: 0)


## 4 · Confronto con Power BI

Stesso periodo del report, gennaio–settembre 2026. I valori attesi, letti nelle schede di Power BI, sono indicati sopra ogni query.

In [8]:
# Atteso da Power BI: ricavi netti 2,20 milioni · ordini 29K · valore medio 85,83 · margine 51,9% · resi 11,7%
!mf query --metrics ricavi_netti,ordini,valore_medio_ordine,margine_pct,tasso_di_reso --group-by metric_time__year --start-time 2026-01-01 --end-time 2026-09-30

✔ Success 🦄 - query completed after 1.30 seconds
metric_time__year      ricavi_netti    ordini    valore_medio_ordine    margine_pct    tasso_di_reso
-------------------  --------------  --------  ---------------------  -------------  ---------------
2026-01-01T00:00:00     2.21962e+06     29123                86.3172       0.518404         0.117351


In [9]:
# Atteso da Power BI: acquirenti 24K · nuovi 17K · di ritorno 10K · quota di ritorno 41,8%
!mf query --metrics clienti_acquirenti,nuovi_clienti,clienti_di_ritorno,quota_clienti_di_ritorno --group-by metric_time__year --start-time 2026-01-01 --end-time 2026-09-30

✔ Success 🦄 - query completed after 1.25 seconds
metric_time__year      clienti_acquirenti    nuovi_clienti    clienti_di_ritorno    quota_clienti_di_ritorno
-------------------  --------------------  ---------------  --------------------  --------------------------
2026-01-01T00:00:00                 24377            17112                 10166                    0.417032


In [10]:
# Atteso da Power BI: Outerwear & Coats 273K · Jeans 251K · Sweaters 169K
!mf query --metrics ricavi_netti --group-by product__category --start-time 2026-01-01 --end-time 2026-09-30 --order -ricavi_netti --limit 5

✔ Success 🦄 - query completed after 1.19 seconds
product__category                ricavi_netti
-----------------------------  --------------
Outerwear & Coats                      264587
Jeans                                  260009
Sweaters                               169895
Suits & Sport Coats                    140298
Fashion Hoodies & Sweatshirts          136898


### Lettura dei risultati

*Da completare dopo l'esecuzione.*